## Question 2
### **Bright Stars Around M67 with ADQL**

In [1]:
# Import modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.table import Table
from astroquery.gaia import Gaia

1.

A colleague is interested in the open cluster Messier 67 (RA = 132.825 deg, Dec = 11.8
deg) and is considering an observation proposal using the 2dF fibre positioner and HERMES
spectrograph (effective for Gaia G < 14) to follow up stars observed with the APOGEE
spectrograph, for which they only have 2MASS catalogue identifiers.

*Write an ADQL query that returns all stars in the Gaia DR3 gaiadr3.gaia source
table within 1 degree of M67 with G < 14 (phot g mean mag) and crossmatches them
with 2MASS (gaiadr1.tmass original valid). You can execute the query on the Gaia
archive website or via astroquery. Include your ADQL query in the notebook and
report the number of returned stars.*

In [21]:
galah_data = Table.read('galah_dr3_allstar_m67_lite.csv')
galah_data['source_id'] = np.array([np.int64(x) for x in galah_data['dr3_source_id']])
gaia_source_ids = galah_data['source_id'].tolist()
gaia_source_ids_table = Table([gaia_source_ids], names=['source_id'])

In [ ]:
query = f"""
SELECT * 
FROM gaiadr3.gaia_source AS gaia
WHERE 1 = CONTAINS(
POINT(132.825, 11.8),
CIRCLE(ra, dec, 1.0))
AND phot_g_mean_mag < 14
JOIN gaiadr1.tmass_original_valid AS gaia1
ON gaia.source_id = gaia1.source_id;
"""

# Upload the source_id table for crossmatching
job = Gaia.launch_job_async(query=query, upload_resource=gaia_source_ids_table, upload_table_name="t1")
gaiadr3_match = job.get_results()

In [ ]:
# within 1 degree of M67
# [RA = 132.825 deg, Dec = 11.8 deg]
# phot_g_mean_mag < 14
# crossmatched with gaiadr1.tmass_original_valid



job = Gaia.launch_job_async(query=query)
gaiadr3_match = job.get_results()

In [12]:
n_stars = len(gaiadr3_match)
print(f"Number of stars in the Gaia DR3 crossmatch: {n_stars}")

1256801741.py: Number of stars in the Gaia DR3 crossmatch: 1029


2.

*Identify stars with:*

-*bad 2MASS photometry (ph qual is not ’AAA’);*

-*non-positive values of Gaia parallax.*

*Apply these quality cuts and report how many stars remain.*

In [ ]:
query2 = f"""
SELECT * 
FROM gaiadr3.gaia_source AS gaia
WHERE 1 = CONTAINS(
POINT(132.825, 11.8),
CIRCLE(gaia.ra, gaia.dec, 1.0))
AND gaia.phot_g_mean_mag < 14
AND gaiadr1.tmass_original_valid.ph_qual IS 'AAA'
AND gaia.parallax > 0
JOIN gaiadr1.tmass_original_valid AS gaia1
ON gaia.source_id = gaia1.source_id;
"""

job2 = Gaia.launch_job_async(query=query2)
gaiadr3_match2 = job2.get_results()

3.

4.